## Notebook 05 — prototipo del Agente Revisor

Objetivo: probar `agente_revisor` (`src_agents/agents/reviewer.py`) de
forma aislada, con casos construidos a mano, antes de conectarlo al
grafo real. Así confirmamos que detecta lo que tiene que detectar sin
depender de gastar cuota de Groq ni de que Ingesta/Analista/Redactor
se ejecuten primero.

El contrato de esta función ya estaba definido antes de escribir una
sola línea de código: `RevisionResultado` (`src_agents/models/state.py`)
dice que `valido=True` solo si todas las cifras del informe coinciden
con el `Analisis`. Este notebook comprueba que la implementación
cumple ese contrato.

In [ ]:
# Celda — setup de rutas (mismo patrón que los notebooks anteriores)
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


### Por qué código y no otra llamada a un LLM

Hay precedente directo en el proyecto para esta decisión:
`src_agents/validation/revisor.py`, el componente de validación de la
fase RAG anterior, ya comparaba cifras por código en vez de con un
modelo — comparar si un número aparece en un texto es una tarea
determinista, y un segundo LLM podría cometer el mismo tipo de error
que se le pide detectar. Además consumiría una llamada más de Groq, y
con el límite diario de tokens ya agotado dos veces durante el
desarrollo (ver `docs/02_README_analyst_agent.md` y
`docs/03_README_workflow_graph.md`), que el Revisor no compita por esa
misma cuota es una ventaja práctica, no solo teórica.

### Paso 1: confirmar que `agente_revisor` importa bien

In [ ]:
from src_agents.agents.reviewer import agente_revisor
from src_agents.models.state import Analisis, ConceptoValor

print("agente_revisor importa correctamente")


### Paso 2: caso base — informe fiel a los datos del Analista

Un `Analisis` con dos `ConceptoValor` y un borrador que menciona ambas
cifras tal cual. Esperamos `valido=True` y `incidencias` vacía.

In [ ]:
analisis_prueba = Analisis(datos=[
    ConceptoValor(concepto="inscritos en el SEPE", valor="1.396",
                  fuente="financiero_informe_1q.docx - Empleo"),
    ConceptoValor(concepto="Nº CONTRATADOS PIL (Diciembre)", valor="16",
                  fuente="financiero_2025_indicadores_control_financiero.xlsx - Centro Formación"),
])

borrador_fiel = (
    "Durante el periodo se registraron 1.396 personas inscritas en el SEPE. "
    "En diciembre, el número de contratados PIL ascendió a 16."
)

resultado_fiel = agente_revisor({"draft": borrador_fiel, "analysis": analisis_prueba})
print(resultado_fiel)
assert resultado_fiel["review"].valido
assert resultado_fiel["review"].incidencias == []


Como se esperaba: las dos cifras están en el texto, así que el Revisor da el informe por válido sin incidencias.

### Paso 3: el Redactor altera una cifra

Mismo `Analisis` de antes, pero el borrador dice `1.400` donde el dato
real es `1.396` — el tipo de error que el Revisor existe para atrapar.
Esperamos `valido=False` y una incidencia señalando esa cifra.

In [ ]:
borrador_alterado = (
    "Durante el periodo se registraron 1.400 personas inscritas en el SEPE. "
    "En diciembre, el número de contratados PIL ascendió a 16."
)

resultado_alterado = agente_revisor({"draft": borrador_alterado, "analysis": analisis_prueba})
print(resultado_alterado)
assert not resultado_alterado["review"].valido
assert len(resultado_alterado["review"].incidencias) == 1


Detectado: la incidencia señala exactamente el concepto y la fuente cuyo valor no aparece en el borrador — suficiente para que una persona revise ese dato concreto sin tener que releer el informe entero.

### Paso 4: el Redactor omite un dato por completo

El borrador no menciona en absoluto la cifra de inscritos en el SEPE.
Para el Revisor esto es indistinguible de una cifra inventada: en
ambos casos, el dato que dio el Analista no aparece en el texto tal
cual, así que el resultado debe ser el mismo tipo de incidencia.

In [ ]:
borrador_incompleto = "En diciembre, el número de contratados PIL ascendió a 16."

resultado_incompleto = agente_revisor({"draft": borrador_incompleto, "analysis": analisis_prueba})
print(resultado_incompleto)
assert not resultado_incompleto["review"].valido


### Paso 5: diferencias de acentos no deben dar falso positivo

El dato de origen lleva tilde (`Educación`) y el borrador lo redacta
sin ella (`Educacion`) — algo plausible si el modelo varía la
ortografía. El Revisor normaliza acentos antes de comparar
(`_quitar_acentos`, mismo enfoque que ya usaba
`src_agents/validation/revisor.py`), así que esto NO debe marcarse
como incidencia.

In [ ]:
analisis_acentos = Analisis(datos=[
    ConceptoValor(concepto="programa evaluado", valor="Educación Infantil",
                  fuente="agencia_innovacion_y_empleo_principales_avances.xlsx"),
])

borrador_acentos = "El programa evaluado corresponde a Educacion Infantil, con buena participación."

resultado_acentos = agente_revisor({"draft": borrador_acentos, "analysis": analisis_acentos})
print(resultado_acentos)
assert resultado_acentos["review"].valido


### Paso 6 (opcional): contra el pipeline real

Monta el grafo completo (Ingesta → Analista → Redactor → Revisor) con
el `agente_revisor` real en vez del stub, sobre los 3 documentos
reales. **Consume cuota de Groq** (Analista y Redactor sí llaman al
modelo) — el Revisor en sí no añade ninguna llamada extra, por diseño.
Igual que en `notebooks_agents/06_workflow_graph_prototype.ipynb`, si
Groq corta por el límite diario no es un fallo de este notebook.

In [ ]:
from groq import APIStatusError
from langgraph.graph import StateGraph, START, END

from src_agents.agents.ingestion import agente_ingesta
from src_agents.agents.analyst import agente_analista
from src_agents.agents.redactor_v1 import agente_redactor
from src_agents.models.state import EstadoPipeline

DATA_DIR = PROJECT_ROOT / "data" / "raw"

grafo_prueba = StateGraph(EstadoPipeline)
grafo_prueba.add_node("ingesta", agente_ingesta)
grafo_prueba.add_node("analista", agente_analista)
grafo_prueba.add_node("redactor", agente_redactor)
grafo_prueba.add_node("revisor", agente_revisor)

grafo_prueba.add_edge(START, "ingesta")
grafo_prueba.add_edge("ingesta", "analista")
grafo_prueba.add_edge("analista", "redactor")
grafo_prueba.add_edge("redactor", "revisor")
grafo_prueba.add_edge("revisor", END)

pipeline_prueba = grafo_prueba.compile()

try:
    resultado_final = pipeline_prueba.invoke({"uploaded_files": [str(DATA_DIR)]})
    print(f"Revisión: {resultado_final['review']}")
except APIStatusError as e:
    print("Groq cortó a mitad del pipeline (cuota agotada o mensaje demasiado grande).")
    print(f"Detalle: {e}")
    print("El Revisor en sí no depende de Groq — el corte es de Analista o Redactor.")


## Conclusión del notebook 05 — prototipo del Agente Revisor

**Objetivo del notebook:** confirmar que `agente_revisor`
(`src_agents/agents/reviewer.py`) cumple el contrato ya definido en
`RevisionResultado` — `valido=True` solo si todas las cifras del
`Analisis` aparecen en el borrador — antes de sustituir el stub
temporal (`agente_revisor_stub`, comentado en
`notebooks_agents/06_workflow_graph_prototype.ipynb`) por la versión
real.

### Lo que se probó, en orden

1. Import limpio de la función, sin depender del resto del pipeline.
2. Caso base: informe fiel a los datos → `valido=True`, sin incidencias.
3. Cifra alterada por el Redactor (`1.396` → `1.400`) → detectada,
   `valido=False`.
4. Dato omitido por completo del informe → detectado igual que una
   alteración.
5. Diferencias de acentos entre el dato de origen y el texto
   redactado → no producen falso positivo.
6. (Opcional) Pipeline real de extremo a extremo, sujeto al límite
   diario de tokens de Groq de Analista y Redactor — el Revisor no
   añade ninguna llamada al modelo.

### Decisión de diseño confirmada

Igual que en `src_agents/validation/revisor.py` (versión anterior de
este componente), la validación es código determinista, no otra
llamada a un LLM — así el Revisor no compite por la misma cuota de
Groq que ya agotan Analista y Redactor.

### Pendiente

- Sustituir `agente_revisor_stub` por `agente_revisor` en
  `notebooks_agents/06_workflow_graph_prototype.ipynb` y en
  `src_agents/graph/workflow.py` cuando se gradúe el grafo completo.
- Si en producción los valores llegan con formatos numéricos distintos
  (separador de miles, decimales) entre lo que devuelve el Analista y
  lo que redacta el modelo, esta comparación por substring literal
  podría dar falsos positivos de incidencia — no se ha visto ese caso
  todavía con datos reales, queda anotado como riesgo a vigilar.